# S17 · Value at Risk: from history and from a bell curve

The question a risk desk answers every morning: **how much could we lose tomorrow?**

We hold ₹10 lakh in five NSE stocks, ₹2 lakh in each: TCS, Infosys, Reliance, ITC and
HDFC Bank. Nobody knows whether tomorrow is up or down, so we ask a narrower question:
on a bad day, how much could we lose? The answer is one rupee number, the **Value at
Risk (VaR)**. Today we compute it two ways: straight from three years of real days, and
from a bell curve. (Notebook 3 adds a third way, Monte Carlo.)

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press the play button on each
  cell, top to bottom, and read the plain-English note above each one.
- New to "quantile" or "the tail"? Open the primer `primers/quantiles_and_tails.md`
  for a ten-minute, picture-first version.
- Already confident with code or with probability? Look for the cells marked
  **Stretch (optional)** near the end.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook downloads stock prices with yfinance, which Colab does not
# always ship. numpy, pandas and matplotlib are already there.
import sys
if "google.colab" in sys.modules:
    !pip install -q yfinance
else:
    print("Not on Colab - assuming the libraries are already installed.")

## Step 1 — get our tools ready

NumPy does maths on long lists of numbers, pandas holds tables of data, and
matplotlib draws charts.

In [ ]:
from pathlib import Path
import numpy as np                 # fast maths on lists of numbers
import pandas as pd                # tables of data
import matplotlib.pyplot as plt    # charts

print("Tools ready.")

## Step 2 — download the prices

Our portfolio is the basket from S16: five well-known NSE stocks. We download three
years of daily closing prices, from January 2023 to December 2025. The dates are
fixed, so you get the same numbers as the slides.

If there is no internet (say, in an exam hall), the cell loads a saved copy of exactly
the same prices instead, so the notebook always runs. It prints which one happened.

In [ ]:
# The five stocks. The '.NS' ending means the National Stock Exchange of India.
tickers = ["TCS.NS", "INFY.NS", "RELIANCE.NS", "ITC.NS", "HDFCBANK.NS"]

# Where the saved copy lives: next to this notebook, or on the course website.
local_copy = Path("../data/nse_five_stocks_2023_2025.csv")
website_copy = ("https://girishmkulkarni.github.io/applied-maths-in-industry-site/"
                "sessions/S17/data/nse_five_stocks_2023_2025.csv")

try:
    import yfinance as yf

    # Daily prices for all five stocks at once. We keep the closing price.
    prices = yf.download(tickers, start="2023-01-01", end="2026-01-01",
                         auto_adjust=True, progress=False)["Close"]

    # Keep our column order and drop any day with a missing price.
    prices = prices[tickers].dropna()

    # An empty table means the download quietly failed.
    if len(prices) < 100:
        raise ValueError("the download came back empty")
    print("Downloaded live prices from Yahoo Finance.")

except Exception as problem:
    print("Could not download (", problem, ") - loading the saved copy instead.")
    if local_copy.exists():
        prices = pd.read_csv(local_copy, index_col=0, parse_dates=True)
    else:
        prices = pd.read_csv(website_copy, index_col=0, parse_dates=True)

print("price table:", prices.shape[0], "days,", prices.shape[1], "stocks")
prices.head()

## Step 3 — turn prices into rupee losses

Three small moves:

1. A **return** is the percentage change in price from one day to the next.
   `pct_change()` works it out for every stock and every day.
2. We put ₹2 lakh in each stock, so each has a **weight** of 0.2 (one fifth of the
   money). `returns @ weights` mixes the five returns into one portfolio return per day,
   the same `@` you met in S16.
3. Risk is about money we *lose*, so we flip the sign and multiply by the ₹10 lakh
   invested. A 1% fall becomes a loss of ₹10,000. From now on, a bigger number is worse.

In [ ]:
# Daily return of each stock: (today's price - yesterday's) / yesterday's.
returns = prices.pct_change().dropna()

# Rs 2 lakh in each of the five stocks, out of Rs 10 lakh.
weights = np.array([0.2, 0.2, 0.2, 0.2, 0.2])
investment = 1_000_000

# One portfolio return per day, flipped into a rupee LOSS.
# A negative return (a fall) becomes a positive loss.
losses = -(returns @ weights) * investment

print("days of data:", len(losses))
print("worst day   :", round(losses.max()), "rupees lost on", losses.idxmax().date())
print("best day    :", round(-losses.min()), "rupees gained on", losses.idxmin().date())

## Step 4 — draw every day as one picture

Before any formula, look at the data. This histogram piles up all our daily losses:
gains on the left, losses on the right. Most days sit near zero, and a few bad days
trail off to the right. Those few days on the right are what risk is about.

In [ ]:
plt.figure(figsize=(9, 5))

# Divide by 1000 so the axis reads in thousands of rupees.
plt.hist(losses / 1000, bins=40, color="#2E75B6")

plt.xlabel("loss that day, thousand rupees  (left = a gain, right = a loss)")
plt.ylabel("number of days")
plt.title("Every trading day, 2023 to 2025, as one picture")
plt.show()

## Step 5 — historical VaR by hand: line up the worst days

The **95% VaR** is the loss that only the worst 5% of days go beyond.

We can find it with no formula at all. Sort the losses from worst to best. 5% of our
739 days is about 37 days, so the 95% VaR sits at roughly the 37th worst day. Here are
the top five and the days around number 37. (Python counts from 0, so the 37th day is
position 36.)

In [ ]:
# Sort the losses, biggest loss first.
worst_first = losses.sort_values(ascending=False)

print("5% of", len(losses), "days is about", round(0.05 * len(losses)), "days\n")

print("The five worst days:")
print(worst_first.head(5).round(0))

print("\nDays number 36, 37 and 38:")
print(worst_first.iloc[35:38].round(0))

## Step 6 — historical VaR in one line

`np.percentile(losses, 95)` does the sorting for us. It returns the value that 95% of
days come in under. It blends the 37th and 38th worst days, so it lands between them.

Then we always check: about 5% of the days should be worse than it.

In [ ]:
# The 95% VaR: 95% of days had a loss smaller than this.
var_hist = np.percentile(losses, 95)

print("Historical 95% VaR:", round(var_hist), "rupees")
print("days worse than that:", (losses > var_hist).sum(), "of", len(losses))

Read it out loud, both ways. They are the same sentence:

- On 95 days out of 100, we lose less than this.
- About 1 day in 20, we lose more than this.

VaR is **not** the worst case. Our worst day, printed in Step 3, was far bigger.

## Step 7 — draw the VaR line on the picture

The same histogram, now with one dashed line at the VaR. Everything to its right (in
red) is the worst 5% of days.

In [ ]:
plt.figure(figsize=(9, 5))

# All the days, then the worst 5% on top in red.
plt.hist(losses / 1000, bins=40, color="#2E75B6")
plt.hist(losses[losses >= var_hist] / 1000, bins=15, color="#C0392B")

# The VaR line.
plt.axvline(var_hist / 1000, color="#C0392B", linestyle="--", linewidth=2.5,
            label="95% VaR")

plt.xlabel("loss that day, thousand rupees")
plt.ylabel("number of days")
plt.title("Only the worst 5% of days lie past the VaR line")
plt.legend()
plt.show()

## Step 8 — parametric VaR: assume a bell curve

A second way. If we are willing to assume the daily losses follow a bell curve, we do
not need to sort anything. A bell curve is described by just two numbers:

- `mu` (μ), the average daily loss, and
- `sigma` (σ), the standard deviation, the typical swing.

95% of any bell curve lies below **μ + 1.645 σ**. So the whole method is:

**VaR = μ + 1.645 × σ**

Our average loss comes out negative, because on an average day we gained a little.

In [ ]:
mu = losses.mean()       # average daily loss
sigma = losses.std()     # typical swing
var_param = mu + 1.645 * sigma

print("mu:", round(mu), "  sigma:", round(sigma))
print("Parametric 95% VaR:", round(var_param), "rupees")
print("Historical 95% VaR:", round(var_hist), "rupees")

At 95% the two answers are close. That closeness is exactly where people stop
checking, so let us check further out.

## Step 9 — the catch: fat tails

Real markets have **fat tails**: very big days happen more often than a bell curve
expects. Two quick checks:

1. Push the confidence to **99%**. For a bell curve the multiplier becomes 2.326.
   Compare it with history.
2. Count the days that lost more than **μ + 3σ**. A bell curve expects only 0.135% of
   days out there.

In [ ]:
# Check 1: the 99% VaR, both ways.
var99_hist = np.percentile(losses, 99)
var99_param = mu + 2.326 * sigma
print("Historical 99% VaR:", round(var99_hist), "rupees")
print("Parametric 99% VaR:", round(var99_param), "rupees")

# Check 2: really big days, beyond 3 standard deviations.
big_days = (losses > mu + 3 * sigma).sum()
bell_curve_expects = len(losses) * 0.00135
print("\ndays beyond 3 sigma:", big_days)
print("a bell curve expects about:", round(bell_curve_expects, 1))

At 99% the bell curve now **understates** the risk, and we had about three times as
many 3-sigma days as it expects. That is the one thing to remember about parametric
VaR: **fast, but it assumes normality.**

### Stretch (optional) — where does 1.645 come from?

Skip this if you are new to code. The number 1.645 is not magic. It is the point on a
standard bell curve (average 0, spread 1) that has 95% of the curve below it. scipy's
`norm.ppf` runs the bell curve backwards: give it a fraction, it gives you the point.
(1.645 is a rounded version of it, so the VaR below can differ by a rupee.)

In [ ]:
from scipy.stats import norm

print("95% point of a bell curve:", round(norm.ppf(0.95), 3))
print("99% point of a bell curve:", round(norm.ppf(0.99), 3))

# The same parametric VaR, written with norm.ppf instead of 1.645.
print("Parametric 95% VaR:", round(mu + norm.ppf(0.95) * sigma), "rupees")

## Your turn

Put more money into Infosys: change the weights to `[0.1, 0.5, 0.2, 0.1, 0.1]` (they
still add up to 1). Recompute the losses and the historical 95% VaR. Does the risk go up
or down? Why might that be?

In [ ]:
# Your code here: new weights, new losses, new historical VaR.

## What you just did

You downloaded three years of real prices, turned them into daily rupee losses for a
₹10 lakh portfolio, and put one number on a bad day: the **95% VaR**, first by lining up
the worst days, then with a bell curve. You also saw where the bell curve goes wrong:
real markets have fatter tails than it allows.

VaR tells you where the bad days start. It says nothing about how bad they get. That is
notebook 2.